# APS 2026 Short Course Tutorial 2: Time-dependent density functional theory (TDDFT)

This tutorial shows how to compute neutral excitation energies of the negatively charged nitrogen-vacancy center (NV$^-$) in diamond using linear-response time-dependent density functional theory (TDDFT) in WEST, which computes the excitation energies by diagonalizing the Liouville superoperator. More details about the TDDFT implementation in WEST can be found in [Jin et al., J. Chem. Theory Comput. 19, 8689–8705 (2023)](https://doi.org/10.1021/acs.jctc.3c00986).

A TDDFT calculation using a local or semi-local exchange-correlation functional is performed in two steps:

1. Perform a ground-state DFT calculation.
2. Compute spin-conserving or spin-flip excitation energies using TDDFT.

## 1. DFT

We perform a ground-state DFT calculation using [Quantum ESPRESSO](https://www.quantum-espresso.org/) (QE).

Download the following files to your working directory:

In [ ]:
%%bash
wget -N -q https://west-code.org/doc/training/nv_diamond_215/tddft/pw.in
wget -N -q http://www.quantum-simulation.org/potentials/sg15_oncv/upf/C_ONCV_PBE-1.2.upf
wget -N -q http://www.quantum-simulation.org/potentials/sg15_oncv/upf/N_ONCV_PBE-1.2.upf

The `pw.in` file is the input for the `pw.x` code:

In [1]:
%%bash
cat pw.in

! NV- in diamond 3x3x3 supercell
! V 0 0 0
&CONTROL
calculation = 'scf'
pseudo_dir = './'
/
&SYSTEM
ibrav = 0
ntyp = 2
nat = 215
tot_charge = -1.0
tot_magnetization = 2.0
nspin = 2
ecutwfc = 60.0
nbnd = 600
/
&ELECTRONS
diago_full_acc = .true.
/
ATOMIC_SPECIES
C  12.01099968  C_ONCV_PBE-1.2.upf
N  14.00699997  N_ONCV_PBE-1.2.upf
CELL_PARAMETERS angstrom
10.704  0.000  0.000
 0.000 10.704  0.000
 0.000  0.000 10.704
K_POINTS gamma
ATOMIC_POSITIONS crystal
N                0.0915286000        0.0915286000        0.0915286000
C                0.0040609500        0.1665643500        0.1665643500
C                0.0837595600        0.2513210000        0.2513210000
C                0.1665643500        0.0040609500        0.1665643500
C                0.2513210000        0.0837595600        0.2513210000
C                0.1665643500        0.1665643500        0.0040609500
C                0.2513210000        0.2513210000        0.0837595600
C                0.0000847900        0.0000847900  

The calculation is spin polarized (i.e., `nspin = 2` and `tot_magnetization = 2.0`), representing the $m_S = +1$ sublevel of the $^3A_2$ many-body state of NV$^-$ in diamond.

We run `pw.x`.

In [ ]:
%%bash
mpirun -n 4 pw.x -i pw.in > pw.out

## 2.1 Perform a spin-conserving TDDFT calculation

We use the `wbse.x` code to compute the excitation energies of triplet states of NV$^-$ in diamond using spin-conserving TDDFT.

Download the following file to your working directory:

In [ ]:
%%bash
wget -N -q https://west-code.org/doc/training/nv_diamond_215/tddft/wbse.in

The `wbse.in` file is the input for the `wbse.x` code:

In [2]:
%%bash
cat wbse.in

input_west:
  outdir: ./

wbse_init_control:
  wbse_init_calculation: S
  solver: TDDFT

wbse_control:
  wbse_calculation: D
  n_liouville_eigen: 2
  n_liouville_times: 16
  trev_liouville: 0.00000001
  trev_liouville_rel: 0.0001


The `n_liouville_eigen: 2` keyword specifies that the two lowest excitation energies are computed.

We run `wbse.x`.

In [ ]:
%%bash
mpirun -n 4 wbse.x -i wbse.in > wbse.out

If the reader does NOT have the computational resources to run the calculation, the output file needed for the next step can be downloaded as:

In [ ]:
%%bash
mkdir -p west.wbse.save
wget -N -q https://west-code.org/doc/training/nv_diamond_215/tddft/wbse.json -O west.wbse.save/wbse.json

The calculated excitation energys (in Rydberg) can be found in a file named `west.wbse.save/wbse.json`.

In [3]:
import json
import numpy as np

Rydberg2eV = 13.6057

fname = "west.wbse.save/wbse.json"

with open(fname, "r") as f:
    j = json.load(f)

vees = np.array(j["exec"]["davitr"][-1]["ev"]) * Rydberg2eV

print("Vertical excitation energies (eV): ", vees)

Vertical excitation energies (eV):  [2.06552077 2.06831693]


We run the `westpp.x` code to analyze the composition of the excited states. We create the input file `westpp.in`:

In [4]:
import yaml

d = {}
d["westpp_control"] = {}
d["westpp_control"]["westpp_calculation"] = "C"
d["westpp_control"]["westpp_n_liouville_to_use"] = 2  # Number of excited states to read from file
d["westpp_control"]["westpp_range"] = [1, 2]  # Excited states to analyze

with open("westpp.in", "w") as f:
    yaml.dump(d, f, sort_keys=False)

In [5]:
%%bash
cat westpp.in

westpp_control:
  westpp_calculation: C
  westpp_n_liouville_to_use: 2
  westpp_range:
  - 1
  - 2


The `westpp_calculation: C` keyword specifies that the code performs a decomposition of the excited states into transitions from occupied to empty Kohn-Sham wavefunctions. Here we consider two excited states as specified by `westpp_range: [1, 2]`.

We run `westpp.x`.

In [ ]:
%%bash
mpirun -n 4 westpp.x -i westpp.in > westpp.out

The output file `westpp.out` would include the following:

```
*-------------* THE PRINCIPLE PROJECTED COMPONENTS *-------------*

#     Exciton :          1 |   Excitation energy :       0.151813
#     Transition_from      |   Transition_to       |    Coeffcient
      1         432        |   1         435       |     0.131741
      2         430        |   2         432       |    -0.968844

#     Exciton :          2 |   Excitation energy :       0.152018
#     Transition_from      |   Transition_to       |    Coeffcient
      1         431        |   1         435       |     0.132307
      2         430        |   2         431       |    -0.968652
```

The first excited state has an excitation energy of 0.151813 Ry (2.066 eV). This excitation has a major contribution by a transition from band 430 (a$_1$ defect orbital) in the spin down channel to band 432 (one of two degenerate e defect orbitals) in the same spin channel. The second excited state has an excitation energy of 0.152018 Ry (2.068 eV). These are the $m_S = +1$ sublevel of the $^3E$ excited states of NV$^-$ in diamond.

## 2.2. Perform a spin-flip TDDFT calculation

Now we run the `wbse.x` code to compute the excitation energies of the singlet states of NV$^-$ in diamond using spin-flip TDDFT.

We modify the `wbse.in` file:

In [6]:
with open("wbse.in", "r") as f:
    d = yaml.load(f, Loader=yaml.SafeLoader)

d["wbse_control"]["n_liouville_eigen"] = 4  # Number of excited states to compute
d["wbse_control"]["l_spin_flip"] = True
d["wbse_control"]["l_spin_flip_kernel"] = True

with open("wbse_sf.in", "w") as f:
    yaml.dump(d, f, sort_keys=False)

In [7]:
%%bash
cat wbse_sf.in

input_west:
  outdir: ./
wbse_init_control:
  wbse_init_calculation: S
  solver: TDDFT
wbse_control:
  wbse_calculation: D
  n_liouville_eigen: 4
  n_liouville_times: 16
  trev_liouville: 1.0e-08
  trev_liouville_rel: 0.0001
  l_spin_flip: true
  l_spin_flip_kernel: true


The `l_spin_flip: True` and `l_spin_flip_kernel: True` keywords specify that the code performs a spin-flip TDDFT calculation with the spin-flip kernel.

We run `wbse.x`.

In [ ]:
%%bash
mpirun -n 4 wbse.x -i wbse_sf.in > wbse_sf.out

If the reader does NOT have the computational resources to run the calculation, the output file needed for the next step can be downloaded as:

In [ ]:
%%bash
mkdir -p west.wbse.save
wget -N -q https://west-code.org/doc/training/nv_diamond_215/tddft/wbse_sf.json -O west.wbse.save/wbse_1.json

The calculated excitation energys (in Rydberg) can be found in a file named `west.wbse.save/wbse_1.json`.

In [8]:
fname = "west.wbse.save/wbse_1.json"

with open(fname, "r") as f:
    j = json.load(f)

vees = np.array(j["exec"]["davitr"][-1]["ev"]) * Rydberg2eV

print("Vertical excitation energies (eV): ", vees)

Vertical excitation energies (eV):  [0.0909167  0.59940457 0.60064678 1.42054327]


The four singlet states are the $m_S = 0$ sublevel of the $^3A_2$ state, the doubly degenerate $^1E$ states, and the $^1A_1$ state.